# AIME — Video Generation with Kling AI (Kling 3.0)

Generates videos from the 57 video-compatible AIME prompts (`Modality` in `{'Video','Both'}`) using **Kling 3.0** via the official [Kling AI Open Platform API](https://kling.ai/document-api).

**API workflow (async job pattern):**
1. Sign a short-lived JWT from your Access Key / Secret Key (`kling.ai/dev/api-key`).
2. `POST /v1/videos/text2video` — submit the job, get back a `task_id`.
3. Poll `GET /v1/videos/text2video/{task_id}` until `task_status` is `succeed`/`failed`.
4. Download the resulting MP4 from `task_result.videos[0].url`.

**Output structure:**
```
dataset/media/videos/Kling/
    P001_amb_<hash>.mp4
    ...
```
A CSV index is written to `dataset/labels/generated_Kling.csv`.

**Prerequisites:**
```bash
pip install pyjwt requests pandas openpyxl tqdm python-dotenv
```

**Credentials required in `.env`:** `KLING_ACCESS_KEY`, `KLING_SECRET_KEY` (see `.env.example`). Get them at https://kling.ai/dev/api-key.
If they are not set, the generation-loop cell will print a warning and skip making any API calls — it will not run
with placeholder/empty credentials.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv
load_dotenv()

KLING_ACCESS_KEY = os.environ.get('KLING_ACCESS_KEY')
KLING_SECRET_KEY = os.environ.get('KLING_SECRET_KEY')

BASE_URL     = 'https://api.klingai.com'
MODEL_NAME   = 'kling-v3'   # Kling 3.0. Other options (per platform docs): kling-v1, kling-v1-5, kling-v1-6,
                            # kling-v2-master, kling-v2-1, kling-v2-1-master, kling-v2-5-turbo, kling-v2-6, kling-v3
ASPECT_RATIO = '16:9'
DURATION     = '5'          # seconds, string per API spec ('5' or '10' for most models; v3 allows 3-15s)

VIDEOS_PER_PROMPT = 1
POLL_INTERVAL_SEC = 10       # docs recommend polling no more than once every ~10s
MAX_POLL_ATTEMPTS = 60       # 60 * 10s = 10 min ceiling per job
MODEL_LABEL = 'Kling'

REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROMPTS_XLS = os.path.join(REPO_ROOT, 'prompts', 'all_prompts.xlsx')
OUTPUT_DIR  = os.path.join(REPO_ROOT, 'dataset', 'media', 'videos', MODEL_LABEL)
INDEX_CSV   = os.path.join(REPO_ROOT, 'dataset', 'labels', f'generated_{MODEL_LABEL}.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(INDEX_CSV), exist_ok=True)

CREDENTIALS_OK = bool(KLING_ACCESS_KEY) and bool(KLING_SECRET_KEY)
print(f'Model     : {MODEL_NAME}')
print(f'Output    : {OUTPUT_DIR}')
print(f'Credentials present: {CREDENTIALS_OK}')
if not CREDENTIALS_OK:
    print('  -> Set KLING_ACCESS_KEY and KLING_SECRET_KEY in .env before running the generation loop.')

In [ ]:
# ── Load prompts ─────────────────────────────────────────────────────────────
import pandas as pd

df_all = pd.read_excel(PROMPTS_XLS, sheet_name='All Prompts (100)')
df_all.columns = df_all.columns.str.strip()

# Video-compatible prompts = Modality in {'Video', 'Both'}
df_vid = df_all[df_all['Modality'].isin(['Video', 'Both'])].copy().reset_index(drop=True)

print(f'Total prompts    : {len(df_all)}')
print(f'Video-compatible : {len(df_vid)}')
print(df_vid.groupby(['Type', 'User Profile']).size().to_string())

In [ ]:
# ── JWT authentication ───────────────────────────────────────────────────────
# Kling signs requests with a short-lived JWT (HS256) built from the Access Key (iss) /
# Secret Key (signing secret), per https://kling.ai/document-api/apiReference/commonInfo
import time
import jwt as pyjwt

def encode_jwt_token(access_key, secret_key):
    headers = {'alg': 'HS256', 'typ': 'JWT'}
    payload = {
        'iss': access_key,
        'exp': int(time.time()) + 1800,  # tokens expire after 30 minutes
        'nbf': int(time.time()) - 5,
    }
    return pyjwt.encode(payload, secret_key, headers=headers)


def auth_headers():
    token = encode_jwt_token(KLING_ACCESS_KEY, KLING_SECRET_KEY)
    return {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

print('JWT helper ready.')

In [ ]:
# ── Generation helpers (submit → poll → download) ───────────────────────────
import requests, hashlib, datetime

def make_filename(prompt_id, prompt_type, index=0):
    ts = datetime.datetime.now().strftime('%Y%m%d%H%M%S%f')
    h  = hashlib.sha256(f'{prompt_id}_{index}_{ts}'.encode()).hexdigest()[:10]
    t  = 'amb' if prompt_type == 'Ambiguous' else 'exp'
    return f'P{prompt_id:03d}_{t}_{h}.mp4'


def submit_text2video(prompt_text):
    """POST /v1/videos/text2video -> task_id"""
    body = {
        'model_name': MODEL_NAME,
        'prompt': prompt_text,
        'aspect_ratio': ASPECT_RATIO,
        'duration': DURATION,
    }
    resp = requests.post(f'{BASE_URL}/v1/videos/text2video', headers=auth_headers(), json=body, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    return data['data']['task_id']


def poll_task(task_id):
    """GET /v1/videos/text2video/{task_id} until succeed/failed. Returns (status, video_url_or_None)."""
    for attempt in range(MAX_POLL_ATTEMPTS):
        resp = requests.get(f'{BASE_URL}/v1/videos/text2video/{task_id}', headers=auth_headers(), timeout=30)
        resp.raise_for_status()
        data = resp.json()['data']
        status = data.get('task_status')
        if status == 'succeed':
            videos = data.get('task_result', {}).get('videos', [])
            return 'ok', (videos[0]['url'] if videos else None)
        if status == 'failed':
            return 'error', data.get('task_status_msg', 'unknown error')
        time.sleep(POLL_INTERVAL_SEC)
    return 'timeout', None


def download_video(url, dest_path):
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

print('Generation helpers ready.')

In [ ]:
# ── Resume: load already-generated index ─────────────────────────────────────
if os.path.exists(INDEX_CSV):
    df_done  = pd.read_csv(INDEX_CSV)
    done_ids = set(zip(df_done['prompt_id'], df_done['vid_index']))
    print(f'Resuming: {len(df_done)} videos already generated.')
else:
    df_done  = pd.DataFrame()
    done_ids = set()
    print('Starting fresh.')

In [ ]:
# ── Main generation loop ─────────────────────────────────────────────────────
from tqdm.notebook import tqdm

records = []

if not CREDENTIALS_OK:
    print('KLING_ACCESS_KEY / KLING_SECRET_KEY not set in .env — skipping API calls.')
    print('Fill in .env (see .env.example) and re-run this cell to actually generate videos.')
else:
    for _, row in tqdm(df_vid.iterrows(), total=len(df_vid), desc=MODEL_LABEL):
        pid         = int(row['ID'])
        ptype       = row['Type']
        prompt_text = str(row['Prompt Text'])

        for vid_idx in range(VIDEOS_PER_PROMPT):
            if (pid, vid_idx) in done_ids:
                continue

            status = 'ok'
            fname  = ''
            try:
                task_id = submit_text2video(prompt_text)
                status, result = poll_task(task_id)
                if status == 'ok' and result:
                    fname = make_filename(pid, ptype, vid_idx)
                    download_video(result, os.path.join(OUTPUT_DIR, fname))
                    done_ids.add((pid, vid_idx))
                elif status == 'ok' and not result:
                    status = 'error'
            except Exception as e:
                print(f'  Error P{pid:03d}: {str(e)[:150]}')
                status = 'error'

            records.append({
                'file_path'   : f'videos/{MODEL_LABEL}/{fname}' if fname else '',
                'prompt_id'   : pid,
                'vid_index'   : vid_idx,
                'prompt_type' : ptype,
                'user_profile': row['User Profile'],
                'modality'    : row['Modality'],
                'category'    : row['Category'],
                'language'    : row['Language'],
                'status'      : status,
                'model'       : MODEL_LABEL,
                'prompt_text' : prompt_text,
            })

if records:
    df_new = pd.DataFrame(records)
    df_out = pd.concat([df_done, df_new], ignore_index=True) if not df_done.empty else df_new
    df_out.to_csv(INDEX_CSV, index=False)
    ok  = (df_out['status'] == 'ok').sum()
    err = (df_out['status'] == 'error').sum()
    print(f'\nDone. Generated: {ok} | Errors: {err}')
    print(f'Index: {INDEX_CSV}')
else:
    print('No records written this run.')

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────
if os.path.exists(INDEX_CSV):
    df_results = pd.read_csv(INDEX_CSV)
    print(f'=== {MODEL_LABEL} Generation Summary ===')
    print(df_results.groupby(['prompt_type', 'status']).size().unstack(fill_value=0).to_string())
    print(f"\nTotal videos saved: {(df_results['status']=='ok').sum()} / {len(df_results)}")
else:
    print('No index CSV yet — run the generation loop first.')